# Updating XML phylogeny names

This notebook parses the XML rather than using regular expressions, to ensure the file remains XML compatible.

In [4]:
from lxml import etree
import os
import pandas as pd

NS = {'OrthoXML': 'http://orthoXML.org/2011/',
      're': 'http://exslt.org/regular-expressions'}

for ns in NS.items():
    etree.register_namespace(*ns)

In [5]:
def map_taxonomy_rewrite(fastoma_result_path, map_fn):
    print('Mapping', fastoma_result_path, '...', end='')
    mapping = pd.read_csv(map_fn, sep='\t', names=['old', 'new']).set_index('old')['new'].to_dict()

    xml = etree.parse(os.path.join(fastoma_result_path, 'FastOMA_HOGs.orthoxml'))

    # map the TaxRange within the HOGs
    for x in xml.iterfind('.//OrthoXML:property[@name=\'TaxRange\']', NS):
        old_label = x.attrib['value']
        if old_label in mapping:
            x.attrib['value'] = mapping[old_label]

    # map names in taxonomy declaration
    for x in xml.iterfind('.//OrthoXML:taxon', NS):
        old_label = x.attrib['name']
        if old_label in mapping:
            x.attrib['name'] = mapping[old_label]

    xml.write(os.path.join(fastoma_result_path, 'FastOMA_HOGs_relabel.orthoxml'), xml_declaration=True, encoding='utf-8')
    print('[DONE]')

In [6]:
map_taxonomy_rewrite(fastoma_result_path='../fastoma_round2_tree_a/result', map_fn='../labelled_trees/tree_a_mapping.tsv')
map_taxonomy_rewrite(fastoma_result_path='../fastoma_round2_tree_b/result', map_fn='../labelled_trees/tree_b_mapping.tsv')
map_taxonomy_rewrite(fastoma_result_path='../fastoma_round2_tree_c/result', map_fn='../labelled_trees/tree_c_mapping.tsv')
map_taxonomy_rewrite(fastoma_result_path='../fastoma_round2_tree_d_unfiltered/result', map_fn='../labelled_trees/tree_d_unfiltered_mapping.tsv')
map_taxonomy_rewrite(fastoma_result_path='../fastoma_round2_tree_e_pfam_filtered/result', map_fn='../labelled_trees/tree_e_pfam_filtered_mapping.tsv')
map_taxonomy_rewrite(fastoma_result_path='../fastoma_round2_tree_f_gopfam_filtered/result', map_fn='../labelled_trees/tree_f_gopfam_filtered_mapping.tsv')

Mapping ../fastoma_round2_tree_a/result ...[DONE]
Mapping ../fastoma_round2_tree_b/result ...[DONE]
Mapping ../fastoma_round2_tree_c/result ...[DONE]
Mapping ../fastoma_round2_tree_d_unfiltered/result ...[DONE]
[DONE]g ../fastoma_round2_tree_e_pfam_filtered/result ...
Mapping ../fastoma_round2_tree_f_gopfam_filtered/result ...[DONE]
